# Algoritmos de Regresión

In [15]:
from pyspark.sql import SparkSession

# 1. Iniciamos Spark en este nuevo notebook
spark = SparkSession.builder.appName("Semana12_Supervisados_Renato").getOrCreate()

# 2. Leemos DIRECTAMENTE los datos que guardamos en el Paso 1
ruta_datos = "/home/jovyan/work/semanas/Semana 10/modelos/datos_etiquetados_kmeans"
df_clusters = spark.read.parquet(ruta_datos)

# 4. Mostramos la tabla para comprobar que TODO está ahí
df_clusters.select("marca", "precio", "ram_gb", "almacenamiento_gb", "prediction").show(10)

+-----+---------+------+-----------------+----------+
|marca|   precio|ram_gb|almacenamiento_gb|prediction|
+-----+---------+------+-----------------+----------+
|    1|1059992.0|    16|            512.0|         0|
|    3|1095386.0|    16|            512.0|         0|
|    0| 789990.0|    16|           1024.0|         0|
|    4|1249990.0|    16|            512.0|         0|
|    0|1078990.0|    16|            512.0|         0|
|    2| 619990.0|    16|            512.0|         0|
|    0| 719990.0|    16|            512.0|         0|
|    0| 719990.0|    16|            512.0|         0|
|    1|3070770.0|    32|           1024.0|         1|
|    0| 679990.0|     8|            512.0|         0|
+-----+---------+------+-----------------+----------+
only showing top 10 rows



In [16]:
from pyspark.ml.feature import VectorAssembler, StandardScaler

# 1. Creamos el VectorAssembler (sin los precios)
assembler_regresion = VectorAssembler(
    inputCols=["ram_gb", "almacenamiento_gb"], 
    outputCol="features_regresion"
)
df_vector_reg = assembler_regresion.transform(df_clusters)

# 2. Escalamos las características
scaler_reg = StandardScaler(inputCol="features_regresion", outputCol="scaledFeatures_regresion")
scaler_model_reg = scaler_reg.fit(df_vector_reg)
df_para_regresion = scaler_model_reg.transform(df_vector_reg)

# 3. Renombramos la variable objetivo Y
df_para_regresion = df_para_regresion.withColumnRenamed("precio", "label_precio")

# =======================================================================
# Borramos la columna prediction del K-Means 
# para que no choque con la de la Regresión Lineal
df_para_regresion = df_para_regresion.drop("prediction")
# =======================================================================

# 4. Dividimos en Entrenamiento (70%) y Prueba (30%)
train_reg, test_reg = df_para_regresion.randomSplit([0.7, 0.3], seed=42)

In [17]:
from pyspark.ml.regression import LinearRegression

# Configurar el modelo de Regresión Lineal
lr_regresion = LinearRegression(
    featuresCol="scaledFeatures_regresion", 
    labelCol="label_precio", 
    maxIter=10
)

# Entrenar el modelo con los datos de entrenamiento
lr_reg_model = lr_regresion.fit(train_reg)

# Hacer las predicciones sobre los datos de prueba
predictions_regresion = lr_reg_model.transform(test_reg)

# Mostrar las predicciones junto al precio real
print("=== COMPARATIVA: PRECIO REAL VS PRECIO PREDICHO ===")
predictions_regresion.select("marca", "label_precio", "prediction").show(10)

=== COMPARATIVA: PRECIO REAL VS PRECIO PREDICHO ===
+-----+------------+------------------+
|marca|label_precio|        prediction|
+-----+------------+------------------+
|    0|    299990.0|1120155.0803120895|
|    0|    409990.0| 853601.8970941561|
|    0|    434990.0| 853601.8970941561|
|    0|    439990.0| 853601.8970941561|
|    0|    459990.0| 853601.8970941561|
|    0|    464990.0| 853601.8970941561|
|    0|    469990.0| 853601.8970941561|
|    0|    515990.0| 853601.8970941561|
|    0|    529990.0| 853601.8970941561|
|    0|    539990.0| 853601.8970941561|
+-----+------------+------------------+
only showing top 10 rows



In [21]:
from pyspark.ml.evaluation import RegressionEvaluator

# Configurar los evaluadores de regresión
evaluator_r2 = RegressionEvaluator(labelCol="label_precio", predictionCol="prediction", metricName="r2")
evaluator_rmse = RegressionEvaluator(labelCol="label_precio", predictionCol="prediction", metricName="rmse")

r2 = evaluator_r2.evaluate(predictions_regresion)
rmse = evaluator_rmse.evaluate(predictions_regresion)

print("==================================================")
print("     EVALUACIÓN DE LA REGRESIÓN (SEMANA 12)       ")
print("==================================================")
print(f"R² (Coeficiente de Determinación): {r2 * 100:.2f}%")
print(f"RMSE (Error promedio del modelo):  {rmse:.4f}")
print("==================================================")

     EVALUACIÓN DE LA REGRESIÓN (SEMANA 12)       
R² (Coeficiente de Determinación): 29.63%
RMSE (Error promedio del modelo):  535780.1605


In [23]:
# Imprimir la intersección (b) y los coeficientes (m)
print(f"Intersección (Precio base): {lr_reg_model.intercept:.4f}")
print(f"Coeficiente de 'rating':    {lr_reg_model.coefficients[0]:.4f}")
print(f"Coeficiente de 'opiniones': {lr_reg_model.coefficients[1]:.4f}")

Intersección (Precio base): 413495.6335
Coeficiente de 'rating':    181767.2220
Coeficiente de 'opiniones': 229776.2183
